Question 29: Lowest Paid
Difficulty: Medium
Link: https://www.interviewquery.com/questions/lowest-paid?playlist=14-days-of-pandas

Problem Description:
===================
Given tables `employees`, `employee_projects`, and `projects`, find the 3 lowest-paid employees that
have completed at least 2 projects.  
  
_Note: incomplete projects will have an end date of`NULL` in the projects table._

**Example:**

**Input:**

`employees` table

Column | Type  
---|---  
`id` | INTEGER  
`first_name` | VARCHAR  
`last_name` | VARCHAR  
`salary` | INTEGER  
`department_id` | INTEGER  
  
`employee_projects` table

Column | Type  
---|---  
`employee_id` | INTEGER  
`project_id` | INTEGER  
  
`projects` table

Column | Type  
---|---  
`id` | INTEGER  
`title` | VARCHAR  
`start_date` | DATE  
`end_date` | DATE  
`budget` | INTEGER  
  
**Output:**

Column | Type  
---|---  
`employee_id` | INTEGER  
`salary` | INTEGER  
`completed_projects` | INTEGER


In [1]:
# Question 29: Lowest Paid
# 
# Find the lowest paid employee in each department.

import pandas as pd

# Mock Data (12 employees, 4 departments, with edge cases)
employees_data = {
    'id': list(range(1, 13)),
    'name': ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank',
             'Grace', 'Hank', 'Ivy', 'Jack', 'Kate', 'Leo'],
    'department': [
        'Engineering', 'Engineering', 'Engineering',  # 3 in Eng
        'Sales', 'Sales',                             # 2 in Sales
        'HR', 'HR', 'HR', 'HR',                       # 4 in HR
        'Marketing',                                  # 1 in Marketing (single person)
        'Engineering', 'Sales',                       # More
    ],
    'salary': [
        80000, 90000, 80000,   # Eng: tied lowest at 80k ✅
        60000, 75000,          # Sales: Eve is lowest
        50000, 55000, 50000, 65000,  # HR: tied lowest at 50k ✅
        70000,                 # Marketing: only 1 person (is the lowest by default)
        85000, 60000,          # More Eng, Sales
    ],
    'hire_date': pd.to_datetime([
        '2020-01-15', '2019-06-01', '2021-03-10',
        '2018-09-20', '2020-11-05',
        '2019-02-14', '2020-07-22', '2021-01-30', '2017-05-10',
        '2022-08-01',
        '2023-01-05', '2022-04-15',
    ])
}
employees = pd.DataFrame(employees_data)
# Edge cases: tied lowest salary, single-person dept, seniority tiebreaker

def solution():
    # Write your solution here
    pass

solution()


### Review of Your Solution

Great job successfully using `transform` to maintain counts without losing the DataFrame structure! However, there's a **critical logical flaw** in the returning step that causes incorrect results (as seen in the output where employee `3` was listed twice).

**1. The Duplication Bug:**
By using `.transform('count')`, you preserved every row. Because Employee `3` completed 2 projects, they have 2 rows containing their data. When you `sort_values` and do `.iloc[0:3]`, you grab those duplicate rows, meaning your top 3 lowest paid spaces are consumed by duplicate entries of the same person!

**2. Unnecessary Merge Execution:**
You run a full merge between `employees` and `employee_projects` immediately. In the real world, this could join millions of records and inflate your dataframe, only for you to filter most of it out 3 steps later when checking `end_date.notnull()`. 

---
### The Fix & Optimization Strategy
1. **Filter Early**: We only care about completed projects. Filter out incomplete projects *before* joining them to employees.
2. **Aggregate Early**: Group the data to count completed projects per employee *first*, filter out those with `< 2`, and only then join the remaining small subset to the `employees` table to grab their salaries.

In [2]:
def solution_optimized(employees, employee_projects, projects):
    # 1. Filter down to ONLY completed projects
    completed_projs = projects[projects['end_date'].notnull()][['id']]
    
    # 2. Join to see which employees are linked to these completed projects
    completed_mapping = pd.merge(employee_projects, completed_projs, left_on='project_id', right_on='id')
    
    # 3. Group by employee and count how many they completed
    counts = completed_mapping.groupby('employee_id').size().reset_index(name='completed_projects')
    
    # 4. Keep only those with 2 or more
    valid_employees = counts[counts['completed_projects'] >= 2]
    
    # 5. Finally, bring in the salaries by merging with a subset of the employees table
    final_df = pd.merge(valid_employees, employees[['id', 'salary']], left_on='employee_id', right_on='id')
    
    # 6. Sort by lowest salary and take the top 3 (.head(3) is cleaner than .iloc[0:3])
    result = final_df.sort_values(by='salary').head(3)[['employee_id', 'salary', 'completed_projects']]
    
    return result 

display(solution_optimized(employees, employee_projects, projects))

NameError: name 'employee_projects' is not defined

---
## Practice Concepts: Aggregation, Filtering, and Merging
Here are some practice questions to reinforce the core mechanics used in this problem.

### Practice Question 1: `.size()` vs `.count()`

**Task:** Given a `tasks` DataFrame with a `completion_date` column, calculate the `total_tasks` and `completed_tasks` for each employee. Create a DataFrame that matches the Expected Output.

**Expected Output:**
```text
   employee_id  total_tasks  completed_tasks
0            1            3                2
1            2            2                0
2            3            1                1
... (remaining rows hidden for brevity)
```

In [3]:
import pandas as pd
import numpy as np

tasks_data = {
    'employee_id': [1, 1, 1, 2, 2, 3],
    'task_id': [101, 102, 103, 104, 105, 106],
    'completion_date': ['2024-01-01', '2024-01-05', np.nan, np.nan, np.nan, '2024-02-01']
}
tasks = pd.DataFrame(tasks_data)

# Write your code here

df = tasks.copy()
# df_tot = df.groupby('employee_id').size().reset_index(name = 'total_task')
# df_comp = df[df['completion_date'].notnull()].groupby('employee_id').size().reset_index(name = 'comp_task')
# df = pd.merge(df_tot,df_comp,on='employee_id',how='left').fillna(0)
# df['comp_task'] = df['comp_task'].astype(int)

df = df.groupby('employee_id').agg(tot_task = ('task_id','size'),comp_task = ('completion_date','count')).reset_index()


display(df)

,employee_id,tot_task,comp_task
0,1,3,2
1,2,2,0
2,3,1,1


### Practice Question 2: Left Anti-Join (Finding Missing Records)

**Task:** We have a new batch of `new_employees` and a table of `assigned_laptops`. Find the `id` and `first_name` of employees who have **not** been assigned a laptop yet.

**Expected Output:**
```text
   id first_name
1   2      Angel
3   4    Jeffrey
```

In [70]:
new_employees_data = {
    'id': [1, 2, 3, 4, 5],
    'first_name': ['Danielle', 'Angel', 'Joshua', 'Jeffrey', 'Jill']
}
new_employees = pd.DataFrame(new_employees_data)

assigned_laptops_data = {
    'laptop_id': ['L1', 'L2', 'L3'],
    'employee_id': [1, 3, 5]  # Angel (2) and Jeffrey (4) are missing
}
assigned_laptops = pd.DataFrame(assigned_laptops_data)

# Write your code here
df = pd.merge(new_employees,assigned_laptops,left_on='id',right_on='employee_id',how='left',indicator=True)

# df = df[df['laptop_id'].isnull()]

display(df)

,id,first_name,laptop_id,employee_id,_merge
0,1,Danielle,L1,1.0,both
1,2,Angel,NaN,NaN,left_only
2,3,Joshua,L2,3.0,both
3,4,Jeffrey,NaN,NaN,left_only
4,5,Jill,L3,5.0,both


### Practice Question 3: Top N per Group

**Task:** Using the updated `employees` DataFrame, find the highest-paid employee in each `department_id`.

**Expected Output:**
```text
   id first_name  last_name  salary  department_id
3   4    Jeffrey  Henderson  243061            200
5   6        Bob      Smith  120000            300
0   1   Danielle     Rhodes  114196            100
... (remaining rows hidden for brevity)
```

In [ ]:
import pandas as pd

# Mock Data for employees dataframe (Multiple employees per department to test grouping!)
employees_data = {
    'id': [1, 2, 3, 4, 5, 6, 7],
    'first_name': ['Danielle', 'Angel', 'Joshua', 'Jeffrey', 'Jill', 'Bob', 'Alice'],
    'last_name': ['Rhodes', 'Mcclain', 'Miller', 'Henderson', 'Johnson', 'Smith', 'Williams'],
    'salary': [114196, 108513, 86579, 243061, 76868, 120000, 95000],
    'department_id': [100, 100, 200, 200, 200, 300, 300],
}
employees = pd.DataFrame(employees_data)

# Write your code here to find the highest-paid employee in each department_id

employees = pd.DataFrame(employees_data)

# Write your code here to find the highest-paid employee in each department_id

df = employees.copy()

df = df[df['salary'] == df.groupby('department_id')['salary'].transform('max')]

display(df)
# display(result_q3)


,id,first_name,last_name,salary,department_id
0,1,Danielle,Rhodes,114196,100
3,4,Jeffrey,Henderson,243061,200
5,6,Bob,Smith,120000,300


### Practice Question 4: Filtering Before Joining

**Task:** You are given `sales` and `products` tables. Calculate the total revenue (`amount`) generated by each `category` **only for sales that occurred in the year 2024**.

**Expected Output:**
```text
      category  revenue
0  Electronics     1200
1     Clothing      150
```

In [2]:
import pandas as pd

sales_data = {
    'sale_id': [1, 2, 3, 4, 5],
    'product_id': [101, 102, 101, 103, 102],
    'sale_date': pd.to_datetime(['2023-12-15', '2024-01-10', '2024-06-20', '2024-08-05', '2023-11-20']),
    'amount': [500, 150, 700, 50, 200]
}
sales = pd.DataFrame(sales_data)

products_data = {
    'product_id': [101, 102, 103],
    'name': ['Laptop', 'Sneakers', 'T-shirt'],
    'category': ['Electronics', 'Clothing', 'Clothing']
}
products = pd.DataFrame(products_data)

# Write your code here
df = pd.merge(sales,products,on='product_id')

df = df[df['sale_date'].dt.year == 2024]

df = pd.merge(sales,products,on='product_id')

df = df.groupby('category')['amount'].sum().reset_index(name = 'revenue')

display(df)


,category,revenue
0,Clothing,400
1,Electronics,1200


### Practice Question 5: Powerful Aggregations (`.agg`)

**Task:** Given an `orders` table, calculate three metrics for each `customer_id` in a single operation:
1. `total_orders`: The total number of orders placed.
2. `total_spend`: The total amount spent.
3. `max_order_value`: The value of their largest single order.

**Expected Output:**
```text
   customer_id  total_orders  total_spend  max_order_value
0            1             3          450              200
1            2             2          800              500
2            3             1           50               50
... (remaining rows hidden for brevity)
```

In [4]:
orders_data = {
    'order_id': [101, 102, 103, 104, 105, 106],
    'customer_id': [1, 2, 1, 3, 2, 1],
    'amount': [150, 300, 100, 50, 500, 200]
}
orders = pd.DataFrame(orders_data)

# Write your code using .agg() here

df = orders.groupby('customer_id').agg(total_orders = ('order_id','count')
                                       ,total_spend = ('amount','sum') 
                                       ,max_order_value = ('amount','max')).reset_index()

display(df)

,customer_id,total_orders,total_spend,max_order_value
0,1,3,450,200
1,2,2,800,500
2,3,1,50,50


### Practice Question 6: Aggregation with Filtering (HAVING clause)

**Task:** Using the `customer_purchases` DataFrame, find the `customer_id`s who have made more than 2 purchases. Return a DataFrame with `customer_id` and their `purchase_count`.

**Expected Output:**
```text
   customer_id  purchase_count
0          101               3
1          103               4
```

In [5]:
import pandas as pd

purchases_data = {
    'customer_id': [101, 102, 101, 103, 103, 104, 103, 101, 103, 105, 102, 104, 105, 101],
    'purchase_amount': [50, None, 30, 40, 70, 200, None, 90, None, 150, 45, None, 60, 100]
}
customer_purchases = pd.DataFrame(purchases_data)

# Write your code here


df = customer_purchases

df = df.groupby('customer_id')['purchase_amount'].count().reset_index(name = 'trial').query('trial >= 1')

display(df)


,customer_id,trial
0,101,4
1,102,1
2,103,2
3,104,1
4,105,2


### Practice Question 7: Counting Unique Values

**Task:** Given a `website_visits` table, calculate the `total_visits` and the number of `unique_pages_visited` for each `user_id`.

**Expected Output:**
```text
   user_id  total_visits  unique_pages_visited
0        1             4                     2
1        2             2                     2
2        3             1                     1
... (remaining rows hidden for brevity)
```

In [15]:
import pandas as pd

visits_data = {
    'user_id': [1, 1, 2, 1, 3, 2, 1, 4, 4, 5, 1, 2, 3, 5, 4, 1, 2, 3, 4, 5],
    'page_url': [
        '/home', '/pricing', '/home', '/about', '/home', '/pricing', '/about',
        '/home', '/checkout', '/home', '/checkout', '/about', '/pricing', '/about',
        '/pricing', '/home', '/home', '/checkout', '/home', '/pricing'
    ]
}
website_visits = pd.DataFrame(visits_data)

# Write your code here
# df = website_visits.groupby('user_id')['page_url'].nunique().reset_index(name = 'unique_visits')

df = website_visits.groupby('user_id').agg(total_visits = ('page_url','size')
                                           ,unique_visits = ('page_url','nunique') ).reset_index()


display(df)


,user_id,total_visits,unique_visits
0,1,6,4
1,2,4,3
2,3,3,3
3,4,4,3
4,5,3,3


### Practice Question 8: Percentage of Total

**Task:** For a `store_sales` DataFrame, calculate what percentage of each `store_id`'s total revenue comes from each `department`. Add a `pct_of_total` column (rounded to 2 decimal places).

**Expected Output:**
```text
   store_id   department  revenue  pct_of_total
0         1    Groceries      500          0.45
1         1    Cosmetics      300          0.27
2         1  Electronics      200          0.18
... (remaining rows hidden for brevity)
```

In [12]:
import pandas as pd

store_sales_data = {
    'store_id': [1, 1, 1, 1, 2, 2, 2, 2, 3, 3, 3, 3, 4, 4, 4, 4],
    'department': [
        'Groceries', 'Cosmetics', 'Electronics', 'Clothing',
        'Groceries', 'Cosmetics', 'Electronics', 'Clothing',
        'Groceries', 'Cosmetics', 'Electronics', 'Clothing',
        'Groceries', 'Cosmetics', 'Electronics', 'Clothing'
    ],
    'revenue': [
        500, 300, 200, 100,  # Store 1
        400, 300, 250, 100,  # Store 2
        600, 200, 150, 250,  # Store 3
        550, 350, 150, 200   # Store 4
    ]
}
store_sales = pd.DataFrame(store_sales_data)

# Write your code here

df = store_sales.copy()

df['pct_of_total'] = (df['revenue']/store_sales.groupby('store_id')['revenue'].transform(sum)).round(2)

# df[]


display(df)


,store_id,department,revenue,pct_of_total
0,1,Groceries,500,0.45
1,1,Cosmetics,300,0.27
2,1,Electronics,200,0.18
3,1,Clothing,100,0.09
4,2,Groceries,400,0.38
5,2,Cosmetics,300,0.29
6,2,Electronics,250,0.24
7,2,Clothing,100,0.10
8,3,Groceries,600,0.50
9,3,Cosmetics,200,0.17


### Practice Question 9: Running / Cumulative Sum

**Task:** Given a `daily_sales` DataFrame, add a `running_total` column that shows the cumulative revenue for each `store_id`, ordered by `sale_date`.

**Expected Output:**
```text
   store_id  sale_date  revenue  running_total
0         1 2024-01-01      200            200
1         1 2024-01-02      150            350
2         1 2024-01-03      300            650
... (remaining rows hidden for brevity)
```

In [37]:
import pandas as pd

daily_sales_data = {
    'store_id': [1, 1, 1, 2, 2],
    'sale_date': pd.to_datetime(['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-01', '2024-01-02']),
    'revenue': [200, 150, 300, 100, 250]
}
daily_sales = pd.DataFrame(daily_sales_data)

# Write your code here

df = daily_sales.copy()

df['running_total'] = df.groupby('store_id')['revenue'].transform('cumsum')

display(df)


,store_id,sale_date,revenue,running_total
0,1,2024-01-01,200,200
1,1,2024-01-02,150,350
2,1,2024-01-03,300,650
3,2,2024-01-01,100,100
4,2,2024-01-02,250,350


### Practice Question 10: Window Ranking with `rank()`

**Task:** Using the `employee_scores` DataFrame, add a `rank` column that ranks each employee **within their department** from highest to lowest score. Use `dense` ranking (no gaps for ties).

**Expected Output:**
```text
   employee_id  department  score  rank
0            1          HR     90     1
1            2          HR     85     2
2            3          HR     85     2
... (remaining rows hidden for brevity)
```

In [49]:
import pandas as pd

scores_data = {
    'employee_id': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
    'department': ['HR', 'HR', 'HR', 'Finance', 'Finance', 'IT', 'IT', 'IT', 'Sales', 'Sales', 'Marketing', 'Marketing', 'Marketing', 'Executive', 'Executive'],
    'score': [90, 85, 85, 95, 70, 88, 92, 88, 80, 95, 82, 91, 82, 98, 94]
}
employee_scores = pd.DataFrame(scores_data)

# Write your code here - compute salary rank per department
employee_scores['rank'] = employee_scores.groupby('department')['score'].rank(method='dense', ascending=False).astype(int)

df = employee_scores.sort_values(by=['department','score'], ascending=[True, False])



display(df)


,employee_id,department,score,rank
13,14,Executive,98,1
14,15,Executive,94,2
3,4,Finance,95,1
4,5,Finance,70,2
0,1,HR,90,1
1,2,HR,85,2
2,3,HR,85,2
6,7,IT,92,1
5,6,IT,88,2
7,8,IT,88,2


### Practice Question 11: Pivot Table / `pd.pivot_table`

**Task:** Given a `region_sales` DataFrame (long format), create a pivot table that shows total `revenue` for each `region` broken down by `quarter` as columns.

**Expected Output:**
```text
quarter   Q1   Q2
region            
East     500  300
West     200  700
... (remaining rows hidden for brevity)
```

In [52]:
import pandas as pd

region_sales_data = {
    'region':  ['East', 'East', 'West', 'West'],
    'quarter': ['Q1',   'Q2',   'Q1',   'Q2'],
    'revenue': [500,    300,    200,    700]
}
region_sales = pd.DataFrame(region_sales_data)

# Write your code here — reshape to wide format using pd.pivot_table

pivot = pd.pivot_table(
    region_sales, 
    values='revenue', 
    index='region', 
    columns='quarter'
)

display(pivot)


quarter,Q1,Q2
region,,
East,500.0,300.0
West,200.0,700.0


### Practice Question 12: `melt` — Wide to Long

**Task:** You have a `scores_wide` DataFrame where Q1, Q2, Q3 are separate columns. Melt it into a long format with columns `student`, `quarter`, `score`.

**Expected Output:**
```text
  student quarter  score
0   Alice      Q1     80
1     Bob      Q1     70
2   Alice      Q2     85
... (remaining rows hidden for brevity)
```

In [ ]:
import pandas as pd

scores_wide_data = {
    'student': ['Alice', 'Bob'],
    'Q1': [80, 70],
    'Q2': [85, 90],
    'Q3': [78, 95]
}
scores_wide = pd.DataFrame(scores_wide_data)

# Write your code here using pd.melt


# display(df_long)


### Practice Question 13: Conditional Column with `np.where` / `pd.cut`

**Task:** Add a `salary_band` column to the `employees` DataFrame:
- `'Low'`  → salary < 90 000
- `'Mid'`  → 90 000 ≤ salary < 150 000
- `'High'` → salary ≥ 150 000

**Expected Output:**
```text
   id  salary salary_band
0   1   76868         Low
1   2   86579         Low
2   3  108513         Mid
... (remaining rows hidden for brevity)
```

In [58]:
import pandas as pd

emp_data = {
    'id':     [1,     2,     3,      4,      5],
    'salary': [76868, 86579, 108513, 114196, 243061]
}
employees = pd.DataFrame(emp_data)

# Write your code here — add a 'salary_band' column

df = employees.copy()

max_sal = df['salary'].max()

# cut_bins = [0,90000,150000,float('inf')]
cut_bins = [0,90000,150000,max_sal]

sal_label = ['Low','Mid','High']

df['sal_band'] = pd.cut(df['salary'],bins=cut_bins,labels = sal_label) 


display(df)


,id,salary,sal_band
0,1,76868,Low
1,2,86579,Low
2,3,108513,Mid
3,4,114196,Mid
4,5,243061,High


### Practice Question 14: Self-Join / Referential Merge

**Task:** Using the `employees` DataFrame that has a `manager_id` column referencing the same table's `id`, create a DataFrame listing each employee's name alongside their manager's name.

**Expected Output:**
```text
  employee_name manager_name
0         Alice          NaN
1           Bob        Alice
2       Charlie        Alice
... (remaining rows hidden for brevity)
```

In [69]:
import pandas as pd
import numpy as np

org_data = {
    'id':         [1,       2,     3,         4],
    'name':       ['Alice', 'Bob', 'Charlie', 'Diana'],
    'manager_id': [np.nan,  1,     1,          2]
}
org = pd.DataFrame(org_data)

# Write your code here — self-join to find each employee's manager name

df = org.copy()

df = pd.merge(df,df,how='left',left_on='manager_id',right_on='id',suffixes=['_employee','_manager'])

df = df[['name_employee','name_manager']]

display(df)


,name_employee,name_manager
0,Alice,NaN
1,Bob,Alice
2,Charlie,Alice
3,Diana,Bob


### Practice Question 15: Rolling Average

**Task:** Given the `stock_prices` DataFrame, add a `3d_avg` column showing the 3-day rolling average of `close_price` per `ticker`. Rows with fewer than 3 previous data points should be `NaN`.

**Expected Output:**
```text
  ticker  close_price      3d_avg
0   AAPL        150.0         NaN
1   AAPL        155.0         NaN
2   AAPL        160.0  155.000000
... (remaining rows hidden for brevity)
```

In [74]:
import pandas as pd

stock_data = {
    'ticker':      ['AAPL', 'AAPL', 'AAPL', 'AAPL', 'GOOG', 'GOOG', 'GOOG'],
    'close_price': [150,    155,    160,    158,    280,    285,    290]
}
stock_prices = pd.DataFrame(stock_data)

# Write your code here — compute 3-day rolling average per ticker

df = stock_prices.copy()

df['3d_avg'] = df.groupby('ticker')['close_price'].transform(lambda x:x.rolling(3).sum()).fillna(0).astype(int)

display(df)


,ticker,close_price,3d_avg
0,AAPL,150,0
1,AAPL,155,0
2,AAPL,160,465
3,AAPL,158,473
4,GOOG,280,0
5,GOOG,285,0
6,GOOG,290,855


### Practice Question 16: String Operations on a Column

**Task:** Given a `contacts` DataFrame with a messy `email` column, use string operations to:
1. Lowercase all emails.
2. Extract the `domain` (everything after `@`).
3. Keep only rows where the domain is `gmail.com`.

**Expected Output:**
```text
   id            email       domain
0   1  alice@gmail.com    gmail.com
2   3  carol@gmail.com    gmail.com
```

In [81]:
import pandas as pd

contacts_data = {
    'id':    [1,                  2,                3,                4],
    'email': ['Alice@Gmail.com', 'Bob@Yahoo.com', 'Carol@Gmail.com', 'Dave@Outlook.com']
}
contacts = pd.DataFrame(contacts_data)

# Write your code here

df = contacts.copy()

df['email'] = df['email'].str.lower()

df['domain'] = df['email'].str.split('@').str[1]

df = df[df['domain'] == 'gmail.com']

display(df)


,id,email,domain
0,1,alice@gmail.com,gmail.com
2,3,carol@gmail.com,gmail.com


### Practice Question 17: Date Component Extraction & Grouping

**Task:** Given a `transactions` DataFrame, compute total `amount` grouped by **year** and **month**.

**Expected Output:**
```text
   year  month  total_amount
0  2023     11           800
1  2023     12           300
2  2024      1           600
... (remaining rows hidden for brevity)
```

In [1]:
import pandas as pd

txn_data = {
    'txn_id': [1, 2, 3, 4, 5, 6],
    'txn_date': pd.to_datetime(['2023-11-05', '2023-12-01', '2023-11-20', '2024-01-15', '2024-01-22', '2024-03-08']),
    'amount':   [500, 300, 300, 200, 400, 450]
}
transactions = pd.DataFrame(txn_data)

# Write your code here — group by year and month

df = transactions.copy()

# df['year','Month'] = [df['txn_date'].dt.year , df['txn_date'].dt.month]

df = df.assign(
    year = df['txn_date'].dt.year,
    month = df['txn_date'].dt.month
)

df = df.groupby(['year','month'])['amount'].sum().reset_index().sort_values(by=['year','month'],ascending =[False,True])

# df['Month'] = df['txn_date'].dt.month

display(df)


,year,month,amount
2,2024,1,600
3,2024,3,450
0,2023,11,800
1,2023,12,300


### Practice Question 19: Exploding Lists into Rows

**Task:** You are given a `users` DataFrame where the `skills` column contains a list of strings for each user. Use the `explode` method to create a separate row for each skill per user.

**Expected Output:**
```text
   user_id   skill
0        1  Python
0        1     SQL
1        2    Java
2        3       R
2        3  Python
```

In [3]:
import pandas as pd

user_data = {
    'user_id': [1, 2, 3],
    'skills': [['Python', 'SQL'], ['Java'], ['R', 'Python']]
}
users = pd.DataFrame(user_data)

# Write your code here - explode the skills list

df = users.copy()

df = df.explode('skills')

display(df)


,user_id,skills
0,1,Python
0,1,SQL
1,2,Java
2,3,R
2,3,Python
